In [1]:
# In current IPython, display lives in IPython.display, not IPython.core.display
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
# This notebook sits in your-code/, so the data folder is one level up
data = pd.read_csv("../data/kg_train.csv", encoding='latin-1')

# Reduce the training set to speed up development.
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("", inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [4]:
# Your code
from sklearn.model_selection import train_test_split

# Later cells expect these exact names: data_train and data_val
# stratify=data['label'] keeps the spam/ham ratio the same in both partitions,
# which matters here because the classes are not balanced (56% ham / 44% spam)
data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data['label'])

print("data_train:", data_train.shape, data_train['label'].value_counts().to_dict())
print("data_val  :", data_val.shape, data_val['label'].value_counts().to_dict())
print()
print("spam ratio -> train %.3f | val %.3f" % (data_train['label'].mean(),
                                               data_val['label'].mean()))

data_train: (800, 2) {0: 446, 1: 354}
data_val  : (200, 2) {0: 112, 1: 88}

spam ratio -> train 0.443 | val 0.440


## Data Preprocessing

In [5]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [6]:
# Your code
import re

def clean_html(text):
    # 1. inline JavaScript / CSS, including whatever sits between the tags
    text = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", text,
                  flags=re.DOTALL | re.IGNORECASE)

    # 2. HTML comments - BEFORE the general tag rule, because a comment can
    #    contain '>' characters that would terminate the tag match early
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # 3. everything else that looks like a tag
    text = re.sub(r"<[^>]+>", " ", text)

    return text

# build the column later cells rely on: data_train['preprocessed_text']
data_train["preprocessed_text"] = data_train["text"].apply(clean_html)
data_val["preprocessed_text"]   = data_val["text"].apply(clean_html)

tags_before = data_train["text"].str.contains(r"<[^>]+>", regex=True).sum()
tags_after  = data_train["preprocessed_text"].str.contains(r"<[^>]+>", regex=True).sum()
print("emails containing HTML tags: %d before -> %d after" % (tags_before, tags_after))

# Note: the step-3 rule also removes anything in angle brackets that is not
# really a tag - e.g. an email address written as <someone@example.com>.
# For spam classification that is acceptable, but it is a real side effect.

emails containing HTML tags: 97 before -> 0 after


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [7]:
# Your code
def clean_text(text):
    # Remove all the special characters  (\W = anything that is not a letter,
    # digit or underscore)
    text = re.sub(r"\W", " ", text)

    # Remove numbers
    text = re.sub(r"\d", " ", text)

    # Remove all single characters  (a lone letter surrounded by spaces)
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text)

    # Remove single characters from the start
    text = re.sub(r"^[a-zA-Z]\s+", " ", text)

    # Remove prefixed 'b'  (left over when bytes objects are cast to str)
    text = re.sub(r"^b\s+", "", text)

    # Substitute multiple spaces with a single space
    text = re.sub(r"\s+", " ", text).strip()

    # Convert to lowercase
    return text.lower()


data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(clean_text)
data_val["preprocessed_text"]   = data_val["preprocessed_text"].apply(clean_text)

print("digits remaining :", data_train["preprocessed_text"].str.contains(r"\d").sum())
print("uppercase        :", data_train["preprocessed_text"].str.contains(r"[A-Z]").sum())
print("double spaces    :", data_train["preprocessed_text"].str.contains(r"  ").sum())
print()
print("sample:", data_train["preprocessed_text"].iloc[0][:200])

# Note: \W is Unicode-aware in Python 3, so accented letters (e, n, o with
# diacritics) count as word characters and survive. Use [^a-zA-Z\s] instead
# of \W if you want strict ASCII.

digits remaining : 0
uppercase        : 0
double spaces    : 0

sample: dear good day hope fine cdear am writting this mail with due respect and heartful of tears since we have not known or met ourselves previously am asking for your assistance ci have will be very glad i


## Now let's work on removing stopwords
Remove the stopwords.

In [8]:
# Your code
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    # the text is already lowercased by the previous cell, so a plain
    # membership test is enough here
    return " ".join(word for word in text.split() if word not in stop_words)


words_before = data_train["preprocessed_text"].apply(lambda t: len(t.split())).sum()

data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(remove_stopwords)
data_val["preprocessed_text"]   = data_val["preprocessed_text"].apply(remove_stopwords)

words_after = data_train["preprocessed_text"].apply(lambda t: len(t.split())).sum()

print("stop words in NLTK's English list:", len(stop_words))
print("total words in train: %d -> %d  (%.1f%% removed)"
      % (words_before, words_after, 100 * (words_before - words_after) / words_before))
print()
print("sample:", data_train["preprocessed_text"].iloc[0][:200])

# Worth knowing for this task: NLTK's list contains "not", "no" and "nor".
# For topic-style classification that is fine - but on a sentiment task,
# removing them would turn "not good" into "good".

stop words in NLTK's English list: 198
total words in train: 174652 -> 105572  (39.6% removed)

sample: dear good day hope fine cdear writting mail due respect heartful tears since known met previously asking assistance ci glad render assistance situation make proposal well known given opportunity would


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [9]:
# Your code
# Cell 10 imported SnowballStemmer, but this section asks for lemmatization,
# so bring in the lemmatizer as well.
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join(lemmatizer.lemmatize(word) for word in text.split())


vocab_before = len(set(w for t in data_train["preprocessed_text"] for w in t.split()))

data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(lemmatize_text)
data_val["preprocessed_text"]   = data_val["preprocessed_text"].apply(lemmatize_text)

vocab_after = len(set(w for t in data_train["preprocessed_text"] for w in t.split()))

print("vocabulary: %d -> %d  (%.1f%% smaller)"
      % (vocab_before, vocab_after, 100 * (vocab_before - vocab_after) / vocab_before))
print()
print("sample:", data_train["preprocessed_text"].iloc[0][:200])

# The reduction is small (~3%) because .lemmatize() treats every word as a
# NOUN by default, so it collapses plurals (agencies -> agency, updates ->
# update) but leaves verb forms alone. Passing pos='v', or tagging parts of
# speech first, would fold far more words together.
#
# The alternative is the stemmer imported in cell 10:
#     snowball.stem(word)
# It would shrink the vocabulary more aggressively, at the cost of producing
# non-words. For a bag-of-words classifier either is defensible, since the
# model only needs tokens to be consistent, not readable.

vocabulary: 29261 -> 28367  (3.1% smaller)

sample: dear good day hope fine cdear writting mail due respect heartful tear since known met previously asking assistance ci glad render assistance situation make proposal well known given opportunity would 


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [10]:
# Your code
from collections import Counter

# label 0 = ham, 1 = spam
ham_words  = Counter(w for t in data_train[data_train["label"] == 0]["preprocessed_text"]
                       for w in t.split())
spam_words = Counter(w for t in data_train[data_train["label"] == 1]["preprocessed_text"]
                       for w in t.split())

print("%-22s %-22s" % ("TOP 10 IN HAM", "TOP 10 IN SPAM"))
for (hw, hn), (sw_, sn) in zip(ham_words.most_common(10), spam_words.most_common(10)):
    print("%-14s %5d    %-14s %5d" % (hw, hn, sw_, sn))

print()
print("distinct words - ham: %d | spam: %d" % (len(ham_words), len(spam_words)))

# --- the words that separate the two classes most sharply ---
print()
print("words far more common in spam than ham:")
for w, n in spam_words.most_common(400):
    if n > 100 and ham_words[w] * 10 < n:
        print("   %-16s spam %4d | ham %3d" % (w, n, ham_words[w]))

# Three artefacts of our cleaning are visible in these lists:
#   nbsp  - the HTML entity &nbsp;  \W stripped the & and ; and left the word
#   u, e  - single letters survive because the "single character" regex only
#           matches a letter with a space on BOTH sides, so it cannot remove
#           two in a row
#   the replacement character - bytes that latin-1 could not map cleanly
# None of them harm the classifier, but a stricter clean would remove them.

TOP 10 IN HAM          TOP 10 IN SPAM        
â                205    money            761
state             97    account          674
president         95    u                631
would             92    bank             615
mr                90    fund             600
ã                 87    e                523
obama             80    transaction      435
percent           80    business         412
call              77    country          401
work              72    nbsp             387

distinct words - ham: 6024 | spam: 24553

words far more common in spam than ham:
   money            spam  761 | ham   6
   account          spam  674 | ham   6
   u                spam  631 | ham  58
   bank             spam  615 | ham   9
   fund             spam  600 | ham   1
   e                spam  523 | ham   0
   transaction      spam  435 | ham   0
   business         spam  412 | ham  27
   nbsp             spam  387 | ham   0
   million          spam  364 | ham  10
   company          s

## Extra features

In [11]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,dear good day hope fine cdear writting mail du...,1,1,1002
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,mr henry kaborethe chief auditor inchargeforei...,0,1,2015
971,Will do.,0,,0,0,0
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,desk dr adamu ismalerauditing accounting manag...,1,1,383
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,dear friend name loi estrada wife mr josephest...,1,1,1475


## How would work the Bag of Words with Count Vectorizer concept?

In [12]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()

# fit_transform on TRAIN - this is where the vocabulary is learned
X_train_bow = count_vectorizer.fit_transform(data_train["preprocessed_text"])

# transform only on VAL - same vocabulary, nothing new learned.
# Using fit_transform here would build a second, different vocabulary, so
# column 500 would mean a different word in each matrix. It would also leak
# validation data into the feature definition.
X_val_bow = count_vectorizer.transform(data_val["preprocessed_text"])

print("X_train_bow:", X_train_bow.shape)
print("X_val_bow  :", X_val_bow.shape, " <- same width, by construction")
print("vocabulary :", len(count_vectorizer.get_feature_names_out()))
print("non-zero   : %.3f%% of cells"
      % (100 * X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1])))

# --- how many words are worth keeping? ---
print()
print("effect of min_df (ignore words appearing in fewer than N emails):")
for m in (1, 2, 5, 10):
    v = CountVectorizer(min_df=m)
    v.fit(data_train["preprocessed_text"])
    print("   min_df=%-3d vocabulary %6d" % (m, len(v.get_feature_names_out())))

# Most of the vocabulary appears in a single email and can teach the
# classifier nothing. min_df is one of the levers to try in the final task.

X_train_bow: (800, 28340)
X_val_bow  : (200, 28340)  <- same width, by construction
vocabulary : 28340
non-zero   : 0.342% of cells

effect of min_df (ignore words appearing in fewer than N emails):


   min_df=1   vocabulary  28340
   min_df=2   vocabulary   5831


   min_df=5   vocabulary   2203
   min_df=10  vocabulary   1209


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [13]:
# Your code
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Vectorize the dataset - fit on TRAIN only, transform VAL with the same vocabulary
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_tfidf   = tfidf_vectorizer.transform(data_val["preprocessed_text"])

# print the shape of the vectorized dataset
print("X_train_tfidf:", X_train_tfidf.shape)
print("X_val_tfidf  :", X_val_tfidf.shape)
print("vocabulary   :", len(tfidf_vectorizer.get_feature_names_out()))

# Same shape as the Bag of Words matrices - the difference is the values.
# Counts become weights: a word appearing in every email is pushed towards
# zero, a word appearing in few is kept at full strength.

X_train_tfidf: (800, 28340)
X_val_tfidf  : (200, 28340)
vocabulary   : 28340


## And the Train a Classifier?

In [14]:
# Your code
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_train = data_train["label"]
y_val   = data_val["label"]

# --- baseline first: what does guessing the majority class get you? ---
baseline = 1 - y_val.mean()
print("baseline (always predict ham): %.4f" % baseline)
print()

# --- compare the two feature sets on the same classifier ---
results = {}
for name, X_tr, X_v in [("Bag of Words", X_train_bow,   X_val_bow),
                        ("TF-IDF",       X_train_tfidf, X_val_tfidf)]:
    model = MultinomialNB()
    model.fit(X_tr, y_train)
    acc = accuracy_score(y_val, model.predict(X_v))
    results[name] = (acc, model, X_v)
    print("%-13s accuracy %.4f   (+%.3f over baseline)" % (name, acc, acc - baseline))

# --- detail for the better one ---
best = max(results, key=lambda k: results[k][0])
acc, model, X_v = results[best]
pred = model.predict(X_v)

print()
print("=== %s, in detail ===" % best)
print(classification_report(y_val, pred, target_names=["ham", "spam"]))
print("confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(y_val, pred))

# Read the confusion matrix, not just the accuracy: the off-diagonal cells say
# WHICH mistake the model makes. Marking a real email as spam (a false
# positive) costs a user far more than letting one scam through, so on this
# task precision on the spam class matters more than raw accuracy.

baseline (always predict ham): 0.5600

Bag of Words  accuracy 0.9650   (+0.405 over baseline)
TF-IDF        accuracy 0.9450   (+0.385 over baseline)

=== Bag of Words, in detail ===
              precision    recall  f1-score   support

         ham       1.00      0.94      0.97       112
        spam       0.93      1.00      0.96        88

    accuracy                           0.96       200
   macro avg       0.96      0.97      0.96       200
weighted avg       0.97      0.96      0.97       200

confusion matrix (rows = true, columns = predicted):
[[105   7]
 [  0  88]]


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [15]:
# Your code